<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/test_image_RAG/Image_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets "chonkie[semantic]" chonkie chromadb "bitsandbytes>=0.46.1" pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.4/386.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3

In [ ]:
import os
import re
import io
import torch
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import login, whoami
from datasets import load_dataset

In [ ]:
hf_token = ""

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

login(token=hf_token)

print(whoami())

{'type': 'user', 'id': '6aab8629f4552c1f3e73bd1f', 'name': 'tdgbcfxdzr', 'fullname': 'Bao Tran', 'isPro': False, 'avatarUrl': '/avatars/6dfa5ced89e7d43ce0c793d6a070f4bd.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'MedGemma', 'role': 'fineGrained', 'createdAt': '2026-09-22T03:51:21.267Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6aab8629f4552c1f3e73bd1f', 'type': 'user', 'name': 'tdgbcfxdzr'}, 'permissions': ['repo.content.read']}]}}}}


In [ ]:
repo_id = "choisongjun/Tropical_Diseases_Repo_1.6"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"Train cases: {len(train_df)} | Test cases: {len(test_df)} | Diseases ({len(DISEASE_LABELS)}): {DISEASE_LABELS}")
train_df.head()

Train cases: 83 | Test cases: 12 | Diseases (8): ['dengue', 'leptospirosis', 'malaria', 'melioidosis', 'neurocysticercosis', 'scrub_typhus', 'typhoid', 'zika']


,case_id,case_text,disease,images
0,PMC10870101_01,A 51-year-old female patient presented to our ...,dengue,[{'image_path': {'bytes': b'RIFF\xa4\x0e\x00\x...
1,PMC6417330_01,A 24-year-old female underwent transcranial su...,dengue,[{'image_path': {'bytes': b'RIFFBM\x00\x00WEBP...
2,PMC10876100_02,"A 43-year-old man, a resident of Uttar Pradesh...",dengue,[{'image_path': {'bytes': b'RIFF`7\x00\x00WEBP...
3,PMC10932498_01,A 17 year old male patient with no previous co...,dengue,[{'image_path': {'bytes': b'RIFF\xe85\x00\x00W...
4,PMC10062082_01,A 28-year-old male presented with complaint of...,dengue,[{'image_path': {'bytes': b'RIFF\xe6\x1b\x00\x...


Image Extractor and format definition

In [ ]:
def extract_pil_images(image_field):

    images = []
    if not image_field.any():
        return images

    for item in image_field:
        if isinstance(item, dict):
            # Output raw image bytes into PIL Image objects of RGB format
            img_path_data = item.get("image_path", {})
            if isinstance(img_path_data, dict) and "bytes" in img_path_data:
                img_bytes = img_path_data["bytes"]
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                images.append(img)
            elif "bytes" in item:
                img = Image.open(io.BytesIO(item["bytes"])).convert("RGB")
                images.append(img)
        # Add pre-loaded PIL Image objects
        elif isinstance(item, Image.Image):
            images.append(item.convert("RGB"))
        # Handle local file paths
        elif isinstance(item, str) and os.path.exists(item):
            images.append(Image.open(item).convert("RGB"))

    return images

# Test image extraction on the first case from the train dataset
sample_images = extract_pil_images(train_df.iloc[0]["images"])
print(f"Extracted {len(sample_images)} image(s) from case {train_df.iloc[0]['case_id']}.")

Extracted 9 image(s) from case PMC10870101_01.


Text Cleaning & Semantic Chunking functions

In [ ]:
from chonkie import (
    SemanticChunker,
    RecursiveRules,
    RecursiveLevel,
    OverlapRefinery,
)

rules = RecursiveRules(levels=[
    RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
    RecursiveLevel(delimiters=["\n"], include_delim="prev"),
    RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    RecursiveLevel(whitespace=True),
])

semantic_chunker = SemanticChunker(
    threshold=0.7,
    chunk_size=400,
    similarity_window=3,
    min_sentences_per_chunk=2,
    skip_window=0,
)

overlap_refinery = OverlapRefinery(
    tokenizer="character",
    context_size=120,
    mode="recursive",
    rules=rules,
    method="prefix",
    merge=True,
)

In [ ]:
# Chunk functions
def clean_text(text: str) -> str:
    text = text.replace("et al.", "et al")
    text = re.sub(r"\[?(?:<[^>]+\.webp>)+(?:-\d+)?\]?", "", text)
    return text

def perform_semantic_chunking(text):
    chunks = semantic_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]

Build Vector store

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import chromadb
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

In [ ]:
COLLECTION_NAME = "tropical_disease_cases"

def build_collection(records, name, rebuild=False):
    if rebuild:
        try:
            chroma_client.delete_collection(name)
        except Exception:
            pass

    collection = chroma_client.get_or_create_collection(name)

    batch = chroma_client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection

def collection_ready(name):
  try:
      collection = chroma_client.get_collection(name)
  except Exception:
      return False
  return collection.count() > 0

def make_records(df, chunk_fn):
    records = []
    for row in df.itertuples():
        chunks = chunk_fn(row.case_text)
        for chunk_id, chunk in enumerate(chunks):
            records.append({
                "document": chunk,
                "metadata": {
                    "case_id": row.case_id,
                    "disease": row.disease,
                    "chunk_id": chunk_id,
                    "total_chunks": len(chunks),
                },
            })
    return records

records = make_records(train_df, perform_semantic_chunking)

In [ ]:
if collection_ready(COLLECTION_NAME):
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print("Reusing existing collection")
else:
    records = make_records(train_df, perform_semantic_chunking)
    collections = build_collection(records, "cases_semantic", rebuild=True)

print(f"ChromaDB collection '{COLLECTION_NAME}' created with {collection.count()} chunks.")

cases_semantic: built 599 chunks
ChromaDB collection 'tropical_disease_cases' created with 599 chunks.


Retrieval Function in RAG

In [ ]:
def retrieve_cases(collection, query_text, k = 3, search_k = 20):
    # Queries ChromaDB at chunk level and aggregates top cases
    res = collection.query(query_texts=[query_text], n_results=search_k)

    best_by_case = {}
    for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
        case_id = meta["case_id"]
        if case_id not in best_by_case or dist < best_by_case[case_id]["distance"]:
            best_by_case[case_id] = {
                "case_id": case_id,
                "disease": meta["disease"],
                "distance": dist,
            }

    return sorted(best_by_case.values(), key=lambda r: r["distance"])[:k]

def attach_case_text(hits, df):
    # Look up the full case text for each retrieved case
    text_by_id = df.set_index("case_id")["case_text"]
    return [{**hit, "text": text_by_id.loc[hit["case_id"]]} for hit in hits]

Load MedGemma Model

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

print("MedGemma successfully loaded.")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma successfully loaded.


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful clinical assistant. Use ONLY the retrieved case reports below as evidence.\n"
    "You MUST answer with exactly one disease name, copied verbatim from this list, and nothing else:\n"
    + "\n".join(f"- {name}" for name in DISEASE_LABELS)
)

def diagnose_image_rag(patient_text, patient_image = None, k = 3, max_new_tokens = 256):
    """
    Executes Multimodal RAG: Retrieves context cases from ChromaDB and passes
    retrieved cases + patient query image to MedGemma for diagnostic reasoning.
    """
    # Step 1: Text-based semantic context retrieval & attach text
    hits = retrieve_cases(collection, query_text=patient_text, k=k)
    retrieved_cases = attach_case_text(hits, train_df)  # <-- Fixed: attaches 'text' key

    context_str = "\n\n".join(
        f"Reference Case #{i+1} (Disease: {res['disease']}): {res['text'][:500]}..."
        for i, res in enumerate(retrieved_cases)
    )

    user_prompt = (
        f"Retrieved Context Cases:\n{context_str}\n\n"
        f"Patient Clinical Presentation:\n{patient_text}\n\n"
        f"Based on the patient's symptoms, clinical image, and retrieved cases, "
        f"provide the final diagnosis and brief clinical justification."
    )

    # Step 2: Format prompt messages for MedGemma
    user_content = []
    if patient_image is not None:
        user_content.append({"type": "image", "image": patient_image})
    user_content.append({"type": "text", "text": user_prompt})

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    # Step 3: Run Model Generation
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return {
        "diagnosis": response,
        "retrieved_cases": retrieved_cases
    }

Run Image RAG Test Demo

In [ ]:
test_sample = test_df.iloc[0]
test_text = test_sample["case_text"]
gt_disease = test_sample["disease"]
test_images = extract_pil_images(test_sample["images"])

query_img = test_images[0] if len(test_images) > 0 else None

print(f"ACTUAL DISEASE: {gt_disease}")
print(f"Patient Text snippet: {test_text[:200]}...")
print(f"Attached query image available: {query_img is not None}\n")

rag_output = diagnose_image_rag(
    patient_text=test_text[:600],
    patient_image=query_img,
    k=3
)

print("RETRIEVED CASES")
for r in rag_output["retrieved_cases"]:
    print(f"- Case ID: {r['case_id']} | Disease: {r['disease']} | Distance: {r['distance']:.3f}")

print("\nMODEL DIAGNOSIS & REASONING")
print(rag_output["diagnosis"])

ACTUAL DISEASE: dengue
Patient Text snippet: A 45-year-old male patient, presented with high-grade continuous fever for 5 days without chills and rigor, persistent headache for 5 days, recurrent episodes of vomiting, and GTCS followed by drowsin...
Attached query image available: True

RETRIEVED CASES
- Case ID: PMC4750307_05 | Disease: dengue | Distance: 0.498
- Case ID: PMC4750307_02 | Disease: dengue | Distance: 0.565
- Case ID: PMC10876100_01 | Disease: dengue | Distance: 0.639

MODEL DIAGNOSIS & REASONING
neurocysticercosis

